# Cookie Cats A/B Test

## Gate Placement and Player Retention

**Notebook:** 04_statistical_testing.ipynb  
**Phase:** 4 — Advanced Analytics & Modelling  
**Source:** BigQuery table `cookie_cats.clean_cookie_cats`, 90,189 players, built from Kaggle "Mobile Games: A/B Testing" (`cookie_cats.csv`), originally from DataCamp project 184  
**Author:** Taofeek Salami  
**Date:** 2026-10-21  

## Test the Retention and Game-Round Differences

This notebook runs the methods selected in `docs/phase_03_method_selection.md`: a two-proportion z-test with a 95% Wald interval for 7-day retention (primary), the same test for 1-day retention under fixed-sequence testing (secondary), a Mann–Whitney U test for game rounds (exploratory guardrail), the minimum detectable difference at 90% power, and the robustness and sensitivity checks. The outcome is read from the 7-day interval using the Decision Rules in `PROJECT_INFO.md`.

In [1]:
import os                            # reads settings from environment variables
from pathlib import Path             # builds file paths from the project root

import numpy as np                   # square roots and resampling for the manual checks and bootstrap
from dotenv import load_dotenv       # loads the .env file into environment variables
from google.cloud import bigquery    # Google's Python client for BigQuery
from scipy.stats import norm         # normal distribution: z critical values and p-values for the manual check
from statsmodels.stats.proportion import (
    proportions_ztest,               # two-proportion z-test (pooled standard error)
    confint_proportions_2indep,      # confidence interval for the difference of two proportions
)
from scipy.stats import mannwhitneyu, rankdata   # rank-based test for game rounds; ranks for its manual check
import pandas as pd                  # tables for the sensitivity results

## Configuration

Settings come from `.env`. The test settings are the values fixed in Decision Rules (`PROJECT_INFO.md`) and Phase 3.

In [2]:
# The notebook runs from notebooks/, so the project root is one folder up.
PROJECT_ROOT = Path.cwd().resolve().parent
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET = os.environ["BQ_DATASET"]
LOCATION = os.environ["BQ_LOCATION"]
TABLE_ID = f"{PROJECT_ID}.{DATASET}.clean_cookie_cats"

ALPHA = 0.05                                  # two-sided significance level (Decision Rules)
Z_CRIT = norm.ppf(1 - ALPHA / 2)              # 1.960: the z-value leaving 2.5% in each tail

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def run_query(sql):
    """Run a query in BigQuery and return the result as a pandas DataFrame."""
    return client.query(sql).to_dataframe()

## Counts by Version

Two-proportion tests need only the number of players and the number retained in each version. Same counts as query 2.1 in `sql/02_business_analysis.sql`.

In [3]:
counts = run_query(f"""
    SELECT
      version,
      COUNT(*)             AS players,
      COUNTIF(retention_1) AS retained_day1,
      COUNTIF(retention_7) AS retained_day7
    FROM `{TABLE_ID}`
    GROUP BY version
    ORDER BY version
""").set_index("version")

print(counts)

# Checks against Phase 2, so any change to the clean table would be caught here.
assert counts.loc["gate_30", "players"] == 44_700 and counts.loc["gate_40", "players"] == 45_489
assert counts["retained_day7"].sum() == 16_781 and counts["retained_day1"].sum() == 40_153
print("Counts match Phase 2")

         players  retained_day1  retained_day7
version                                       
gate_30    44700          20034           8502
gate_40    45489          20119           8279
Counts match Phase 2


## 4.1 7-Day Retention (primary)

Δ = 7-day retention for gate_40 minus 7-day retention for gate_30. The z-test uses the pooled standard error; the 95% Wald interval uses each version's own rate. The decision is read from the interval, as Decision Rules state.

In [4]:
def compare_retention(metric_column, table=None):
    """Two-proportion z-test and 95% Wald interval for gate_40 minus gate_30.
    Uses the main counts unless a filtered counts table is passed in."""
    table = counts if table is None else table
    x40, n40 = table.loc["gate_40", metric_column], table.loc["gate_40", "players"]
    x30, n30 = table.loc["gate_30", metric_column], table.loc["gate_30", "players"]

    # gate_40 is listed first so the statistic and interval refer to gate_40 − gate_30.
    z_stat, p_value = proportions_ztest(count=[x40, x30], nobs=[n40, n30], alternative="two-sided")
    ci_low, ci_high = confint_proportions_2indep(
        count1=x40, nobs1=n40, count2=x30, nobs2=n30,
        method="wald", compare="diff", alpha=ALPHA,
    )
    return {
        "rate_gate_30": x30 / n30,
        "rate_gate_40": x40 / n40,
        "difference": x40 / n40 - x30 / n30,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "z": z_stat,
        "p_value": p_value,
    }


def decision_from_interval(ci_low, ci_high):
    """Apply the Decision Rules in PROJECT_INFO.md to the 95% interval for Δ."""
    if ci_low > 0:
        return "Move the gate to level 40 (L > 0: 7-day retention is higher for gate_40)"
    if ci_high < 0:
        return "Keep the gate at level 30 (U < 0: 7-day retention is lower for gate_40)"
    return "Collect more evidence (the interval includes 0)"


day7 = compare_retention("retained_day7")

print(f"7-day retention: gate_30 {day7['rate_gate_30']:.4%}, gate_40 {day7['rate_gate_40']:.4%}")
print(f"Difference (gate_40 − gate_30): {day7['difference'] * 100:.3f} percentage points")
print(f"95% Wald interval: {day7['ci_low'] * 100:.3f} to {day7['ci_high'] * 100:.3f} percentage points")
print(f"z = {day7['z']:.3f}, two-sided p = {day7['p_value']:.4f}")
print(f"\nDecision under the Decision Rules: {decision_from_interval(day7['ci_low'], day7['ci_high'])}")

7-day retention: gate_30 19.0201%, gate_40 18.2000%
Difference (gate_40 − gate_30): -0.820 percentage points
95% Wald interval: -1.328 to -0.312 percentage points
z = -3.164, two-sided p = 0.0016

Decision under the Decision Rules: Keep the gate at level 30 (U < 0: 7-day retention is lower for gate_40)


### Manual Check

Recalculates the z statistic and the interval from the formulas, to confirm the library results.

In [5]:
x40, n40 = counts.loc["gate_40", "retained_day7"], counts.loc["gate_40", "players"]
x30, n30 = counts.loc["gate_30", "retained_day7"], counts.loc["gate_30", "players"]
p40, p30 = x40 / n40, x30 / n30
difference = p40 - p30

# Interval: unpooled standard error, from each version's own rate.
se_unpooled = np.sqrt(p40 * (1 - p40) / n40 + p30 * (1 - p30) / n30)
manual_ci = (difference - Z_CRIT * se_unpooled, difference + Z_CRIT * se_unpooled)

# Test: pooled standard error, from the combined rate the null hypothesis assumes.
p_pooled = (x40 + x30) / (n40 + n30)
se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n40 + 1 / n30))
manual_z = difference / se_pooled
manual_p = 2 * norm.sf(abs(manual_z))   # sf = 1 − cdf; doubled for a two-sided test

print(f"Standard error (unpooled, interval): {se_unpooled:.6f}")
print(f"Standard error (pooled, test):       {se_pooled:.6f}")
print(f"Manual interval: {manual_ci[0] * 100:.3f} to {manual_ci[1] * 100:.3f} pp")
print(f"Manual z = {manual_z:.3f}, p = {manual_p:.4f}")

assert np.isclose(manual_ci[0], day7["ci_low"]) and np.isclose(manual_ci[1], day7["ci_high"])
assert np.isclose(manual_z, day7["z"]) and np.isclose(manual_p, day7["p_value"])
print("Manual calculation matches statsmodels")

Standard error (unpooled, interval): 0.002592
Standard error (pooled, test):       0.002592
Manual interval: -1.328 to -0.312 pp
Manual z = -3.164, p = 0.0016
Manual calculation matches statsmodels


**Result:**  
7-day retention was 18.20% for gate_40 and 19.02% for gate_30, a difference of −0.82 percentage points (95% interval −1.33 to −0.31; z = −3.16, p = 0.0016). The interval lies entirely below zero, so under the Decision Rules the outcome is to keep the gate at level 30. The difference is statistically significant at α = 0.05. Reading it as caused by the gate position depends on random assignment (A-002), which is untested. The pooled and unpooled standard errors are equal to six decimal places, so the test and the interval agree.

## 4.2 1-Day Retention (secondary)

Fixed-sequence testing (Phase 3): 1-day retention is tested at α = 0.05 only because the 7-day interval excludes zero. The result supports interpretation and does not change the decision.

In [6]:
# Fixed-sequence gate: the secondary test is read as significant or not only if
# the primary interval excludes zero; otherwise it is reported as supporting information.
primary_significant = day7["ci_low"] > 0 or day7["ci_high"] < 0

day1 = compare_retention("retained_day1")

print(f"Primary (7-day) interval excludes zero: {primary_significant}")
print(f"\n1-day retention: gate_30 {day1['rate_gate_30']:.4%}, gate_40 {day1['rate_gate_40']:.4%}")
print(f"Difference (gate_40 − gate_30): {day1['difference'] * 100:.3f} percentage points")
print(f"95% Wald interval: {day1['ci_low'] * 100:.3f} to {day1['ci_high'] * 100:.3f} percentage points")
print(f"z = {day1['z']:.3f}, two-sided p = {day1['p_value']:.4f}")

if primary_significant:
    significant = day1["p_value"] < ALPHA
    print(f"\nTested at α = {ALPHA}: {'statistically significant' if significant else 'not statistically significant'}")
else:
    print("\nNot tested: the primary interval includes zero, so this is supporting information only")

Primary (7-day) interval excludes zero: True

1-day retention: gate_30 44.8188%, gate_40 44.2283%
Difference (gate_40 − gate_30): -0.591 percentage points
95% Wald interval: -1.239 to 0.058 percentage points
z = -1.784, two-sided p = 0.0744

Tested at α = 0.05: not statistically significant


**Resul:**  
1-day retention was 44.23% for gate_40 and 44.82% for gate_30, a difference of −0.59 percentage points (95% interval −1.24 to +0.06; z = −1.78, p = 0.074). Tested at α = 0.05 under fixed-sequence testing, the difference is not statistically significant: the data are consistent with no difference at day 1 and with a drop of up to about 1.2 points. The point estimate is in the same direction as the 7-day result. It does not change the decision.

## 4.3 Game Rounds (guardrail, exploratory)

Mann–Whitney U test: do players in one version tend to play more rounds than players in the other? The effect size is the probability of superiority, the chance that a randomly chosen gate_40 player played more rounds than a randomly chosen gate_30 player, with ties counted as half. Exploratory under Decision Rules: outside the error-rate control, and it does not block the decision.

In [7]:
# One row per player: the test needs every value, not just counts.
rounds = run_query(f"SELECT version, sum_gamerounds FROM `{TABLE_ID}`")
r40 = rounds.loc[rounds["version"] == "gate_40", "sum_gamerounds"].to_numpy()
r30 = rounds.loc[rounds["version"] == "gate_30", "sum_gamerounds"].to_numpy()

# method="asymptotic" uses the normal approximation with a correction for tied
# values; exact methods are impractical for 90,000 players with many ties.
# With gate_40 first, U counts the player pairs where the gate_40 player played
# more rounds, plus half of the tied pairs.
u_stat, p_value_rounds = mannwhitneyu(r40, r30, alternative="two-sided", method="asymptotic")
prob_superiority = u_stat / (len(r40) * len(r30))

print(f"Players: gate_30 {len(r30):,}, gate_40 {len(r40):,}")
print(f"Medians: gate_30 {np.median(r30):.0f}, gate_40 {np.median(r40):.0f}")
print(f"U = {u_stat:,.0f}, two-sided p = {p_value_rounds:.4f}")
print(f"Probability of superiority (gate_40 over gate_30): {prob_superiority:.4f}")

Players: gate_30 44,700, gate_40 45,489
Medians: gate_30 17, gate_40 16
U = 1,009,027,050, two-sided p = 0.0502
Probability of superiority (gate_40 over gate_30): 0.4962


### Manual Check

U can be calculated from ranks: rank all players together (tied values share the average rank), sum the gate_40 ranks, and subtract the smallest possible sum.

In [8]:
# rankdata gives tied values the average of the ranks they span.
all_ranks = rankdata(np.concatenate([r40, r30]))
rank_sum_40 = all_ranks[: len(r40)].sum()
manual_u = rank_sum_40 - len(r40) * (len(r40) + 1) / 2   # subtract the minimum possible rank sum

print(f"Manual U = {manual_u:,.0f}")
assert np.isclose(manual_u, u_stat)
print("Manual U matches scipy")

Manual U = 1,009,027,050
Manual U matches scipy


**Result:**  
The game-round distributions are very similar: medians of 17 (gate_30) and 16 (gate_40), and a probability of superiority of 49.6%, meaning a randomly chosen gate_40 player played more rounds than a randomly chosen gate_30 player 49.6% of the time, against 50% if neither version tended to play more (Mann–Whitney U, p = 0.050). The data show at most a very small tendency for gate_40 players to play fewer rounds. This comparison is exploratory under Decision Rules, so it is reported and does not affect the decision.

## 4.4 Minimum Detectable Difference

The smallest true difference in 7-day retention this sample would detect with 90% power at α = 0.05 (two-sided), using the gate_30 rate as the baseline. It describes the design's sensitivity and uses no outcome results.

In [9]:
POWER = 0.90                                   # chosen in Phase 3
z_power = norm.ppf(POWER)                      # 1.282: z-value for 90% power

baseline = counts.loc["gate_30", "retained_day7"] / counts.loc["gate_30", "players"]
n30 = counts.loc["gate_30", "players"]
n40 = counts.loc["gate_40", "players"]

# Standard error of a difference between two proportions at the baseline rate.
se_design = np.sqrt(baseline * (1 - baseline) * (1 / n30 + 1 / n40))
mdd = (Z_CRIT + z_power) * se_design

print(f"Baseline 7-day retention (gate_30): {baseline:.4%}")
print(f"z for α/2 = {Z_CRIT:.3f}, z for 90% power = {z_power:.3f}")
print(f"Minimum detectable difference: {mdd * 100:.3f} percentage points "
      f"({mdd / baseline:.2%} relative to the baseline)")

Baseline 7-day retention (gate_30): 19.0201%
z for α/2 = 1.960, z for 90% power = 1.282
Minimum detectable difference: 0.847 percentage points (4.45% relative to the baseline)


**Result:**  
With these group sizes, α = 0.05 (two-sided) and 90% power, the minimum detectable difference in 7-day retention is 0.85 percentage points (4.45% relative to gate_30's 19.02%). The observed gap of −0.82 points is close to this value: a true gap of that size would be detected about 88% of the time, and gaps much smaller than 0.85 points would often be missed. Because significant results near the detection limit tend to overstate the true effect, the full 95% interval (−1.33 to −0.31 points) is the better summary of the effect size than the point estimate alone.

## 4.5 Robustness Check: Bootstrap Interval

A second 95% interval for the 7-day difference that does not rely on the normal approximation. For a yes/no outcome, resampling a version's players with replacement is equivalent to drawing its retained count from a binomial distribution with the observed rate, so 10,000 resamples are drawn that way. The interval is the 2.5th to 97.5th percentile of the resampled differences. Reported whatever its result; it does not replace the main interval.

In [10]:
N_BOOT = 10_000
rng = np.random.default_rng(seed=2026)   # fixed seed so the result can be reproduced

x40, n40 = counts.loc["gate_40", "retained_day7"], counts.loc["gate_40", "players"]
x30, n30 = counts.loc["gate_30", "retained_day7"], counts.loc["gate_30", "players"]

# Each draw is the retained count from resampling all of a version's players.
boot_diff = rng.binomial(n40, x40 / n40, N_BOOT) / n40 - rng.binomial(n30, x30 / n30, N_BOOT) / n30
boot_low, boot_high = np.percentile(boot_diff, [2.5, 97.5])

print(f"Bootstrap 95% interval: {boot_low * 100:.3f} to {boot_high * 100:.3f} percentage points")
print(f"Wald 95% interval:      {day7['ci_low'] * 100:.3f} to {day7['ci_high'] * 100:.3f} percentage points")
print(f"Share of resamples with gate_40 lower: {(boot_diff < 0).mean():.2%}")

Bootstrap 95% interval: -1.332 to -0.316 percentage points
Wald 95% interval:      -1.328 to -0.312 percentage points
Share of resamples with gate_40 lower: 99.83%


**Result:**  
The bootstrap 95% interval for the 7-day difference is −1.33 to −0.32 percentage points, within 0.004 points of the Wald interval at both ends, so the main interval does not depend on the normal approximation. The share of resamples in which gate_40 was lower (99.83%) shows that resampling almost never reverses the direction; it is not a probability that gate_40 is worse.

## 4.6 Sensitivity Checks

The 7-day retention test and the game-round comparison rerun after excluding the flagged extreme player, the zero-round players, or both. These groups are defined by game rounds, which were recorded after assignment, so the filtered comparisons are less protected by randomisation than the main analysis. They show whether the conclusion depends on these players; they do not replace the main analysis.

In [11]:
# Player-level data with the cleaning flags, so each filter uses the flags from Phase 1.
players = run_query(f"""
    SELECT version, sum_gamerounds, retention_7, is_extreme_gamerounds, is_zero_gamerounds
    FROM `{TABLE_ID}`
""")

filters = {
    "All players (main analysis)": np.ones(len(players), dtype=bool),
    "Excluding the extreme player": ~players["is_extreme_gamerounds"],
    "Excluding zero-round players": ~players["is_zero_gamerounds"],
    "Excluding both": ~players["is_extreme_gamerounds"] & ~players["is_zero_gamerounds"],
}

rows = []
for label, keep in filters.items():
    subset = players[keep]
    subset_counts = subset.groupby("version").agg(
        players=("version", "size"),
        retained_day7=("retention_7", "sum"),
    )
    result = compare_retention("retained_day7", subset_counts)

    s40 = subset.loc[subset["version"] == "gate_40", "sum_gamerounds"].to_numpy()
    s30 = subset.loc[subset["version"] == "gate_30", "sum_gamerounds"].to_numpy()
    u, p_rounds = mannwhitneyu(s40, s30, alternative="two-sided", method="asymptotic")

    rows.append({
        "analysis": label,
        "players": len(subset),
        "diff_pp": round(result["difference"] * 100, 3),
        "ci_low_pp": round(result["ci_low"] * 100, 3),
        "ci_high_pp": round(result["ci_high"] * 100, 3),
        "p_7day": round(result["p_value"], 4),
        "decision": decision_from_interval(result["ci_low"], result["ci_high"]).split(" (")[0],
        "prob_superiority": round(u / (len(s40) * len(s30)), 4),
        "p_rounds": round(p_rounds, 4),
    })

sensitivity = pd.DataFrame(rows)
print(sensitivity.to_string(index=False))

                    analysis  players  diff_pp  ci_low_pp  ci_high_pp  p_7day                  decision  prob_superiority  p_rounds
 All players (main analysis)    90189   -0.820     -1.328      -0.312  0.0016 Keep the gate at level 30            0.4962    0.0502
Excluding the extreme player    90188   -0.818     -1.326      -0.310  0.0016 Keep the gate at level 30            0.4962    0.0509
Excluding zero-round players    86195   -0.812     -1.341      -0.284  0.0026 Keep the gate at level 30            0.4969    0.1163
              Excluding both    86194   -0.810     -1.339      -0.282  0.0026 Keep the gate at level 30            0.4969    0.1177


**Result:**  
The 7-day difference stays between −0.81 and −0.82 percentage points in every analysis, and every interval excludes zero, so the decision is unchanged when the extreme player, the zero-round players, or both are excluded. For game rounds, the probability of superiority stays between 49.6% and 49.7%, but the p-value rises from 0.050 to 0.116 when zero-round players are excluded: without them, the small tendency for gate_40 players to play fewer rounds is weaker, so part of it is associated with gate_40's higher share of zero-round players (4.52% against 4.33%). These filtered comparisons use groups defined after assignment and do not replace the main analysis.

## Summary

| Analysis | Result |
|----------|--------|
| 7-day retention (primary) | 18.20% gate_40 vs 19.02% gate_30; −0.82 pp (95% interval −1.33 to −0.31), p = 0.0016 |
| Decision under Decision Rules | Keep the gate at level 30 |
| 1-day retention (secondary) | −0.59 pp (95% interval −1.24 to +0.06), p = 0.074; not statistically significant |
| Game rounds (exploratory) | Probability of superiority 49.6%, Mann–Whitney p = 0.050; very similar distributions |
| Minimum detectable difference | 0.85 pp at 90% power |
| Bootstrap interval | −1.33 to −0.32 pp |
| Sensitivity checks | Decision unchanged in all three filtered analyses |

The causal reading, that moving the gate to level 40 lowered 7-day retention, depends on random assignment (A-002), which cannot be confirmed from the data.